In [ ]:
#Importing required libraries
import traceback
import pandas as pd
import re
import pdfplumber
from collections import namedtuple
from calendar import month_name
import fitz
import dataUtil as DU
import logging

In [ ]:
import sys
print(sys.executable)
print(sys.version)
print(sys.version_info)

In [ ]:
import types
def imports():
    for name, val in globals().items():
        if isinstance(val, types.ModuleType):
            yield val.__name__
list(imports())

In [ ]:
import pkg_resources
import types
def get_imports():
    for name, val in globals().items():
        if isinstance(val, types.ModuleType):
            # Split ensures you get root package, 
            # not just imported function
            name = val.__name__.split(".")[0]

        elif isinstance(val, type):
            name = val.__module__.split(".")[0]
            
        # Some packages are weird and have different
        # imported names vs. system/pip names. Unfortunately,
        # there is no systematic way to get pip names from
        # a package's imported name. You'll have to add
        # exceptions to this list manually!
        poorly_named_packages = {
            "PIL": "Pillow",
            "sklearn": "scikit-learn"
        }
        if name in poorly_named_packages.keys():
            name = poorly_named_packages[name]
            
        yield name
imports = list(set(get_imports()))

# The only way I found to get the version of the root package
# from only the name of the package is to cross-check the names 
# of installed packages vs. imported packages
requirements = []
for m in pkg_resources.working_set:
    if m.project_name in imports and m.project_name!="pip":
        requirements.append((m.project_name, m.version))

for r in requirements:
    print("{}=={}".format(*r))

In [ ]:
#Import PDF file from path
InputDate = '2026-07-20'
currentLCR = f'912 Playbook {InputDate}.pdf'
file=f"C:\\Users\\thomas2.DESKTOP-F01LKSM\\Google Drive Streaming\\My Drive\\ReadProjects\\Neural Matrix Investment\\LCR\\{currentLCR}"
file=f"C:\\Users\\Asus\\My ive\\projects\\Neural Matrix Investment\\LCR\\{currentLCR}"
file=f"I:\\My Drive\\ReadProjects\\Neural Matrix Investment\\LCR\\{currentLCR}"
# file="G:\\My Drive\\projects\\Neural Matrix Investment\\LCR\\LCR Playbook 2022-06-06.pdf"
# file = 'D:\\Users\\thomas2\\Atwmchoi2010_GDrive\\Projects\\Neural Matrix Investment\\LCR\\LCR Playbook 2022-03-21.pdf'
#   Ubuntu/WSL file path location
file=f"/mnt/i/My Drive/ReadProjects/Neural Matrix Investment/LCR/{currentLCR}"
print('Processing LCR:',file)
with fitz.open(file) as doc:
    text = ""
    text_full = ""
    c=0
    for page in doc:
        c += 1
        if c < 3: continue
#         text = page.getText()
        text = page.get_text()
        print(f"==> page: {c}", text)
        text_full += text
print(text_full)

In [ ]:
iterates = iter(text_full.split('\n'))
data_foo = [['','','','','']]

#Basic columns to be read from PDF
col= ["Type", "Symbol", "Trend", "Status","Expiration", "PnC", "L_Strike" , "H_Strike", "Entry", "Target", "Stop"]
data_f = []
for x in col:
    data_f.append('')
data_foo = [data_f]
df = pd.DataFrame(data_foo,columns=col)

In [ ]:
df.head()


In [ ]:
c=0
num = 0
for line in iterates:
    num+=1
    print(line.find("Bond ETF Trade & Maintenance Suggestions"))
    if line.find("Bond ETF Trade & Maintenance Suggestions") >= 0:
        break
print(line)
print(num)

In [ ]:
iterates = iter(text_full.split('\n'))
Type_name = None
Stock_name_tmp = None
Status = None
Trend = None
line_num = -1
row_number = -1
checkk = 0
for line in iterates:
    print(line, checkk, num)
    checkk+=1
    if checkk+1 > num:
        try:
            if line.find("Trade & Maintenance Suggestions") > 0:
                ch1 = line.find("Trade & Maintenance Suggestions")
                Type_name = line[:ch1].strip()
                print(f'Type_name = {Type_name}')
            else:
                if line.strip() == '' or line.find("Aggressive Traders:") > -1:
                    continue
                if line.find("Trend") == -1 and re.search("\[\w*\]", line):
                    Stock_name_tmp = (re.findall(r"\[\w*\]", line)[0][1:-1])
                    line_num = 0
                    print(f'Stock_name is {Stock_name_tmp}')
                else:
                    if line_num == 0:
                        Trend = re.findall(r"Investor|Trader", line)[0]
                        Status = re.findall(r"\[\S*\]", line)[0][1:-1]
                        line_num=1
                    else:
                        if (line.find("Aggressive Traders:") > -1):
                            pass
                        elif (line.find("Trend") > 0):
                            Trend = re.findall(r"Investor|Trader", line)[0]
                            Status = re.findall(r"\[\S*\]", line)[0][1:-1]
                        elif ((line.find("STO") > -1) or (line.find("BTO") > -1)):
                            row_number+= 1
                            df.loc[len(df.index)] = data_f
                            df['Type'][row_number] = Type_name
                            df['Symbol'][row_number] = Stock_name_tmp
                            df['Trend'][row_number] = Trend
                            df['Status'][row_number] = Status
                            decission = re.findall(r"Put|Call", line)[0][0]
                            df['PnC'][row_number] = decission
                            print('** Find Exp Date **', line)
                            exp_pattern = r'\((\d{1,2}/\d{1,2}/\d{2,4})\s*exp\)'
                            matches = re.findall(exp_pattern, line)
                            print(" new found exp date: ", matches)
                            exp_date = matches[0]
                            # if line.count(")") > 1:
                            #     split_line = line.split(')', 1)
                            #     print('split_line == ', split_line)
                            #     date2 = line.split(')', 1)[1]
                            #     print('date2== ', date2)
                            #     exp_date = re.findall(r"\(...*\)", date2)[0][1:-4]
                            # else:
                            #     exp_date = re.findall(r"\(...*\)", line)[0][1:-4]
                            df['Expiration'][row_number] = exp_date
                            price = line.split('(')[0]
                            matches = re.finditer(r"\d+(\.)?\d+", price)
                            print(f'matches : {matches}')
                            tmp_price = []
                            for m in matches:
                                tmp_price.append(m[0])
                            l_price = None
                            h_price = tmp_price[0]
                            if len(tmp_price) == 2:
                                l_price = tmp_price[0]
                                h_price = tmp_price[1]
                            df["L_Strike"][row_number] = l_price
                            df["H_Strike"][row_number] = h_price
                        elif (line.find("Entry @ ") > -1):
                            entry = line.split(',')[0]
                            mtch_entry = re.finditer(r"\d+(\.)?\d+", entry)
                            ent = []
                            for n in mtch_entry:
                                ent.append(n[0])
                            df["Entry"][row_number] = ent[0]
                            target = line.split(',')[1]
                            mtch_trgt = re.finditer(r"\d+(\.)?\d+", target)
                            trgt = []
                            for z in mtch_trgt:
                                trgt.append(z[0])
                            df["Target"][row_number] = trgt[0]
                        elif (line.find("Stop") > -1):
                            if line.count("@") > 1:
                                stop = line.split('@',2)[2]
                            else:
                                stop = line.split('@')[1]
                            mtch_stop = re.finditer(r"\d+(\.)?\d+", stop)
                            stp = []
                            for q in mtch_stop:
                                stp.append(q[0])
                            df["Stop"][row_number] = stp[0]
                        else:
                            pass
        except Exception as e:
            print(f'Exception on line:{line} => {price}')
print(df)

In [ ]:
df.head()

In [ ]:
df = df[:-1]
df["Low"] = ''
df["High"] = ''
df["Low-High"] = ''

#Creating Low-High variable
for index, row in df.iterrows():
    try:
        df["Low"][index] = ('+' + df["L_Strike"][index] + df["PnC"][index]) if df["L_Strike"][index] else df["Low"][index]
        df["High"][index] = ('-' + df["H_Strike"][index] + df["PnC"][index]) if df["H_Strike"][index] else df["High"][index]
        df["Low-High"][index] = df["Low"][index] + df["High"][index]
    except: pass

In [ ]:
df.tail(5)

In [ ]:
df = df.fillna('')
for index, row in df.iterrows():
    print('r=',index)
    print('Sy=',row["Symbol"],',','Ent=',row["Entry"],',','St=',row["Stop"],row["PnC"],',','Exp=',row["Expiration"])
    try:
        if (df["Stop"][index] or df["Symbol"][index]!=df["Symbol"][index+1] or df["Expiration"][index]!=df["Expiration"][index+1]):
            nstop = df["Stop"][index]
        else:
            if df["PnC"][index+1] == "C":
                nstop = df["Stop"][index+1]
            else:
                nstop = df["Entry"][index+1]
        print('** ', index, ' - ', nstop)
        nstop = df["Stop"][index] if (df["Stop"][index] or df["Symbol"][index]!=df["Symbol"][index+1] or df["Expiration"][index]!=df["Expiration"][index+1]) else df["Stop"][index+1] if df["PnC"][index+1] == "C" else df["Entry"][index+1] 
#         nstop = df["Entry"][index+1] if (not df["Stop"][index] and df["Symbol"][index] == df["Symbol"][index+1]) else df["Stop"][index]
        print('**** ', index, ' - ', nstop)
    except: pass

In [ ]:
df = df.fillna('')
df['Entry_Sign'] = df['PnC'].apply(lambda x: '<' if x == 'P' else '>')
df['Target_Sign'] = df['PnC'].apply(lambda x: '>' if x == 'P' else '<')
df['Stop_Sign'] = df['Entry_Sign']

#Filling all empty stop values from next row Empty values for same stock
for index, row in df.iterrows():
    try:
        df["Stop"][index] = df["Stop"][index] if (df["Stop"][index] or df["Symbol"][index]!=df["Symbol"][index+1] or df["Expiration"][index]!=df["Expiration"][index+1]) else df["Stop"][index+1] if df["PnC"][index+1] == "C" else df["Entry"][index+1] 
#         df["Stop"][index] = df["Entry"][index+1] if (not df["Stop"][index] and df["Symbol"][index] == df["Symbol"][index+1]) else df["Stop"][index]
    except: pass

In [ ]:
df[df["Trend"] == "Investor"]

In [ ]:
#Filling all empty stop values from next row Empty values for same stock
for index, row in df.iterrows():
    try:
        df["Stop_Sign"][index] = df["Stop_Sign"][index-1] if (df["Type"][index] == df["Type"][index-1] and df["Symbol"][index] == df["Symbol"][index-1] and df["Trend"][index] == df["Trend"][index-1]) else df["Entry_Sign"][index]
    except: pass

In [ ]:
for index, row in df.iterrows():
    try:
        df["Entry_Sign"][index] = df["Entry_Sign"][index-1] if (df["Type"][index] == df["Type"][index-1] and df["Symbol"][index] == df["Symbol"][index-1] and df["Trend"][index] == df["Trend"][index-1]) else df["Entry_Sign"][index]
    except: pass

In [ ]:
for index, row in df.iterrows():
    try:
        df["Target_Sign"][index] = df["Target_Sign"][index-1] if (df["Type"][index] == df["Type"][index-1] and df["Symbol"][index] == df["Symbol"][index-1] and df["Trend"][index] == df["Trend"][index-1]) else df["Target_Sign"][index]
    except: pass

In [ ]:
df['Date'] = InputDate
df

In [ ]:
#Arranging columns as required in output file
df = df[["Date","Type", "Trend", "Symbol", "Status", "Expiration", "PnC", "Low-High", "L_Strike" , "H_Strike", "Entry_Sign", "Entry", "Target_Sign", "Target", "Stop_Sign", "Stop"]]

#Seperating the Investor and Trader Files
df_Investor = df[df["Trend"]=="Investor"]
df_Investor.reset_index(drop = True, inplace = True)
df_Trader = df[df["Trend"]=="Trader"]
df_Trader.reset_index(drop = True, inplace = True)

In [ ]:
df_Investor

In [ ]:
#Adding 2 empty rows to differentiate between Investor and Trader
# df_Investor = df_Investor.append(pd.Series(), ignore_index=True)
# df_Investor = df_Investor.append(pd.Series(), ignore_index=True)

In [ ]:
#Appending back both the Investor and Trader dataframes, one below the other
final_df = pd.concat([df_Investor, df_Trader], ignore_index=True, sort=False)
final_df = final_df.fillna('')

#Exports the final dataframe to txt file
# final_df.to_csv('check222.txt', sep='\t', index = None)
# final_df.to_csv(file.replace('.pdf', '.csv'), sep=',', index = None)
final_df
final_df[final_df["Trend"] == "Investor"]

In [ ]:
from dotenv import load_dotenv
import logging
import numpy as np

In [ ]:
tf = file.replace('.pdf', '.csv')
print(tf)
final_df.to_csv(tf, sep=',', index = None)

In [ ]:
final_df = pd.read_csv(tf)
final_df

In [ ]:
final_df["Date"] = pd.to_datetime(final_df["Date"])
final_df["Expiration"] = pd.to_datetime(final_df["Expiration"])
final_df = final_df.replace(r'^\s*$', np.nan, regex=True)
# final_df['L_Strike'] = final_df['L_Strike'].astype(float)
final_df = final_df.astype({'L_Strike':'float','H_Strike':'float','Target':'float','Entry':'float','Stop':'float'})

final_df.info()

In [ ]:
load_dotenv("DB_Config.env") #Check path for env variables
logging.getLogger().setLevel(logging.DEBUG)

In [ ]:
final_df

In [ ]:
opt_tbl = "ETF_Options"
trd_DB = "Trading"
DU.StoreEOD(final_df, trd_DB, opt_tbl)

In [ ]:
pwd

In [ ]:
final_df.head()

In [ ]:
final_df.to_csv("final_etf_df.csv", index=False)